# ECE 232E: Structured Memory Networks and RICR

This notebook runs against either self-contained course package: `panini_2wiki_100` or `panini_musique_100`. Corpus embeddings and indexes are supplied; you will analyze the GSW network, compare retrieval methods, and implement Reasoning Inference Chain Retrieval (RICR).

## 0. Colab setup

Open this notebook from the public GitHub repository. The next cell clones the complete package automatically. If you use a Google Drive copy, change only `PACKAGE_ROOT`. A GPU is needed only for the optional Qwen cells.

In [ ]:
from pathlib import Path
import subprocess

PACKAGE_ROOT = Path('/content/panini-course-project')
if not (PACKAGE_ROOT / 'manifest.json').exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/YigitTurali/panini-course-project.git',
        str(PACKAGE_ROOT),
    ], check=True)
assert (PACKAGE_ROOT / 'manifest.json').exists(), 'Set PACKAGE_ROOT to the release folder'

%pip install -q -r {PACKAGE_ROOT / 'requirements-colab.txt'}

import sys
sys.path.insert(0, str(PACKAGE_ROOT))

## 1. Inspect the dataset and GSW network

In [ ]:
from panini_course import CoursePackage
from panini_course.graph import build_entity_projection, build_native_gsw_graph

package = CoursePackage(PACKAGE_ROOT)
print(package.manifest['counts'])
questions = package.questions('public')
questions[0]

In [ ]:
# Start with a small slice; build the complete network for your report.
native_graph = build_native_gsw_graph(package.gsw_paths()[:25])
entity_graph = build_entity_projection(native_graph)
print('native:', native_graph.number_of_nodes(), native_graph.number_of_edges())
print('projection:', entity_graph.number_of_nodes(), entity_graph.number_of_edges())

# TODO: components, GCC, degree distributions, centrality, PageRank,
# clustering, assortativity, and one multi-hop path visualization.

## 2. Compare dense, TF-IDF, BM25, and hybrid retrieval

In [ ]:
from panini_course import BM25Index, DenseIndex, QueryEmbeddingStore, TfidfIndex

query_store = QueryEmbeddingStore.load(
    PACKAGE_ROOT / 'embeddings/query_embeddings.npy',
    PACKAGE_ROOT / 'embeddings/query_ids.json',
    PACKAGE_ROOT / 'embeddings/queries.jsonl',
)
qa_dense = DenseIndex.load(
    PACKAGE_ROOT / 'indices/qa_qwen3_8b_ip.faiss',
    PACKAGE_ROOT / 'indices/qa_ids.json', source='qa_dense',
)
qa_tfidf = TfidfIndex.load(
    PACKAGE_ROOT / 'indices/qa_tfidf.npz',
    PACKAGE_ROOT / 'indices/qa_tfidf_vectorizer.joblib',
    PACKAGE_ROOT / 'indices/qa_ids.json', source='qa_tfidf',
)
qa_bm25 = BM25Index.load(
    PACKAGE_ROOT / 'indices/qa_bm25.joblib',
    PACKAGE_ROOT / 'indices/qa_ids.json', source='qa_bm25',
)

query = questions[0]['question']
dense_hits = qa_dense.search(query_store.get(query), 10)
tfidf_hits = qa_tfidf.search(query, 10)
bm25_hits = qa_bm25.search(query, 10)
[(hit.item_id, hit.score) for hit in dense_hits[:3]]

In [ ]:
from panini_course import reciprocal_rank_fusion

hybrid_hits = reciprocal_rank_fusion(
    [dense_hits, tfidf_hits, bm25_hits], top_k=10
)
[(hit.item_id, hit.score, hit.metadata['fusion_sources']) for hit in hybrid_hits[:5]]

# TODO: evaluate QA and entity retrieval, then implement paper-style
# BM25 entity expansion + dense QA retrieval + reranking.

## 3. Question decomposition

Use the public reviewed subset first. Then load the fine-tuned decomposer from `models/model_config.json`. Parse `<ENTITY_Qn>` references as dependency edges and identify independent or parallel retrieval chains. Load only one Qwen model at a time on free-tier Colab.

In [ ]:
import json

model_config = json.loads((PACKAGE_ROOT / 'models/model_config.json').read_text())
validation_decompositions = package.decompositions()
example_id, example_plan = next(iter(validation_decompositions.items()))
example_id, example_plan

In [ ]:
# Optional GPU cell; run it in a separate runtime stage and save outputs.
# from panini_course.qwen_models import QwenDecomposer
# decomposer = QwenDecomposer(
#     model_config['decomposer']['model'],
#     PACKAGE_ROOT / model_config['decomposer']['prompt'],
# )
# predicted_plan = decomposer.decompose(query)
# del decomposer
# import gc, torch; gc.collect(); torch.cuda.empty_cache()

## 4. Implement RICR

Complete `identify_retrieval_components` and `run_panini_ricr` in `panini_course/ricr.py`. Execute each connected retrieval DAG in topological order. Combine multi-parent beams before substitution, group answer entities only at intermediate hops, keep QA-level alternatives at the final hop, and deduplicate evidence from every surviving final beam. The package uses document-namespaced local entity IDs; the analysis-only reconciliation graph must not enter RICR. All query embeddings reached by the required deterministic runs are supplied; do not load an embedding model.

In [ ]:
from panini_course import Candidate, run_panini_ricr

def retrieve_and_score(instantiated_question: str, top_k: int):
    # TODO: dual retrieval -> format QA candidates -> Qwen reranker.
    # Return one Candidate per QA record, retaining all answer_names,
    # document-namespaced answer_ids, answer_role_states, and score.
    raise NotImplementedError

# result = run_panini_ricr(example_plan, retrieve_and_score,
#                           original_question=query, beam_width=5,
#                           candidates_per_hop=15)

## 5. Evaluation and ablations

Report Recall@k, MRR, supporting-document/QA recall, complete-chain recovery, answer EM/F1, latency, and evidence size. Compare retrieval backends, beam width, intermediate entity grouping, multi-parent thresholding, last-hop versus geometric-mean scoring, gold versus predicted decompositions, and increasing distractor counts. Format all-final-beam evidence with answer role/states and use the supplied PANINI one-shot answer prompt without adding an N/A instruction.

In [ ]:
from panini_course.metrics import exact_match, recall_at_k, reciprocal_rank, token_f1

# TODO: accumulate one JSONL result per question and summarize by question type.